# RiskPY Validation Notebook

Smoke-tests the v0.2.5 C++ APIs: FactorModel, LossTriangle, RateAnalyzer, FourierTransform, MonteCarlo.

In [ ]:
import riskpy
from riskpy import (
    FactorModel, LossTriangle, RateAnalyzer,
    FourierTransform, MonteCarloSimulator, ActuarialMath,
)
print("riskpy", riskpy.__version__)
print("exports", [n for n in riskpy.__all__ if n != "UnderwritingApp"])

In [ ]:
model = FactorModel(1000.0)
model.add_multiplier("state", "FL", 3.0)
model.add_numeric_band_multiplier("age", 16, 25, 2.0)
print("premium", model.calculate({"state": "FL", "age": 19.0}))

In [ ]:
t = LossTriangle()
t.add_origin_year(2019, [1000.0, 1500.0, 1800.0, 2000.0])
t.add_origin_year(2020, [1200.0, 1800.0, 2160.0])
t.add_origin_year(2021, [1400.0, 2100.0])
t.add_origin_year(2022, [1600.0])
print("LDFs", t.get_development_factors())
print("ultimates", t.get_ultimate_losses())
print("IBNR", t.get_ibnr_reserves(), "total", sum(t.get_ibnr_reserves()))

In [ ]:
pmf = FourierTransform.compound_poisson_pmf([0.0, 1.0], 2.0, grid_size=64)
print("P(S=0..5)", [round(p, 6) for p in pmf[:6]])
print("mean S", sum(k * p for k, p in enumerate(pmf)))

In [ ]:
sim = MonteCarloSimulator(trials=5000, seed=42)
losses = sim.simulate_aggregate_loss(5.0, 8.0, 1.0)
print("MC mean", sum(losses) / len(losses))